In [14]:
from dotenv import load_dotenv
import os
import math
import time
import requests
import pandas as pd



## NY EMS DATA Ingest 

In [ ]:
DATASET_ID = "76xm-jjuj"  #NYC EMS Incident Dispatch Data
BASE_URL = f"https://data.cityofnewyork.us/resource/{DATASET_ID}.json"

# --- adjust these two dates as needed (keep it small at first Team) ---
START_DATE = "2024-01-01T00:00:00.000"
END_DATE   = "2024-04-01T00:00:00.000"


In [ ]:
COLS = [
    "cad_incident_id",
    "incident_datetime",
    "initial_call_type",
    "initial_severity_level_code",
    "final_call_type",
    "final_severity_level_code",
    "valid_dispatch_rspns_time_indc",
    "dispatch_response_seconds_qy",
    "valid_incident_rspns_time_indc",
    "incident_response_seconds_qy",
    "incident_travel_tm_seconds_qy",
    "held_indicator",
    "incident_disposition_code",
    "borough",
    "zipcode",
]

def socrata_pull_all(
    where: str,
    select_cols: list[str],
    limit: int = 50000,
    sleep_s: float = 0.2,
) -> pd.DataFrame:
    """pulls all rows matching a where clause using socrata pagination."""
    frames = []
    offset = 0

    # we request in chunks until a chunk returns 0 rows
    while True:
        params = {
            "$select": ",".join(select_cols),
            "$where": where,
            "$limit": limit,
            "$offset": offset,
        }
        r = requests.get(BASE_URL, params=params, timeout=60)
        r.raise_for_status()
        chunk = r.json()

        if not chunk:
            break

        frames.append(pd.DataFrame(chunk))
        offset += limit
        time.sleep(sleep_s)

        if offset % (limit * 5) == 0:
            print(f"pulled {offset:,} rows so far...")

    if not frames:
        return pd.DataFrame(columns=select_cols)

    df = pd.concat(frames, ignore_index=True)
    return df

where_clause = (
    f"incident_datetime >= '{START_DATE}' "
    f"AND incident_datetime < '{END_DATE}'"
)

df_raw = socrata_pull_all(where=where_clause, select_cols=COLS, limit=50000)
print("rows:", len(df_raw))
df_raw.head()

pulled 250,000 rows so far...
rows: 385234


,cad_incident_id,incident_datetime,initial_call_type,initial_severity_level_code,final_call_type,final_severity_level_code,valid_dispatch_rspns_time_indc,dispatch_response_seconds_qy,valid_incident_rspns_time_indc,incident_response_seconds_qy,incident_travel_tm_seconds_qy,held_indicator,incident_disposition_code,borough,zipcode
0,240010001,2024-01-01T00:00:03.000,INJURY,5,INJURY,5,Y,53,Y,393,340,N,93,QUEENS,11365
1,240010002,2024-01-01T00:00:14.000,CARD,3,CARD,3,Y,84,Y,430,346,N,82,BRONX,10457
2,240010003,2024-01-01T00:00:18.000,STNDBM,8,STNDBY,8,N,0,N,NaN,433,N,82,BROOKLYN,11224
3,240010004,2024-01-01T00:00:45.000,RESPIR,4,RESPIR,4,Y,1485,Y,1934,449,Y,82,MANHATTAN,10033
4,240010005,2024-01-01T00:00:55.000,BURNMI,4,BURNMI,4,Y,39,Y,1315,1276,N,82,BROOKLYN,11208


In [ ]:
df_raw["incident_datetime"] = pd.to_datetime(df_raw["incident_datetime"], errors="coerce")

num_cols = [
    "dispatch_response_seconds_qy",
    "incident_response_seconds_qy",
    "incident_travel_tm_seconds_qy",
]
for c in num_cols:
    df_raw[c] = pd.to_numeric(df_raw[c], errors="coerce")


df_raw["zipcode"] = (
    df_raw["zipcode"]
    .astype("string")
    .str.extract(r"(\d{5})", expand=False)
)

os.makedirs("../data/raw", exist_ok=True)

out_path = "../data/raw/nyc_ems_2024q1_raw.parquet"
df_raw.to_parquet(out_path, index=False)

print("saved:", out_path)

saved: ../data/raw/nyc_ems_2024q1_raw.parquet


In [18]:
df_check = pd.read_parquet("../data/raw/nyc_ems_2024q1_raw.parquet")
print(df_check.shape)
df_check.head()

(385234, 15)


,cad_incident_id,incident_datetime,initial_call_type,initial_severity_level_code,final_call_type,final_severity_level_code,valid_dispatch_rspns_time_indc,dispatch_response_seconds_qy,valid_incident_rspns_time_indc,incident_response_seconds_qy,incident_travel_tm_seconds_qy,held_indicator,incident_disposition_code,borough,zipcode
0,240010001,2024-01-01 00:00:03,INJURY,5,INJURY,5,Y,53,Y,393.0,340.0,N,93,QUEENS,11365
1,240010002,2024-01-01 00:00:14,CARD,3,CARD,3,Y,84,Y,430.0,346.0,N,82,BRONX,10457
2,240010003,2024-01-01 00:00:18,STNDBM,8,STNDBY,8,N,0,N,NaN,433.0,N,82,BROOKLYN,11224
3,240010004,2024-01-01 00:00:45,RESPIR,4,RESPIR,4,Y,1485,Y,1934.0,449.0,Y,82,MANHATTAN,10033
4,240010005,2024-01-01 00:00:55,BURNMI,4,BURNMI,4,Y,39,Y,1315.0,1276.0,N,82,BROOKLYN,11208


In [19]:
df = df_raw.copy()

df_usable = df[
    (df["valid_incident_rspns_time_indc"] == "Y") &
    (df["incident_response_seconds_qy"].notna()) &
    (df["incident_response_seconds_qy"] > 0) &
    (df["incident_response_seconds_qy"] <= 7200) &
    (df["incident_datetime"].notna()) &
    (df["borough"].notna())
].copy()

os.makedirs("../data/processed", exist_ok=True)
out_path = "../data/processed/nyc_ems_2024q1_usable.parquet"
df_usable.to_parquet(out_path, index=False)

print("raw rows:", len(df_raw))
print("usable rows:", len(df_usable))
print("saved:", out_path)

raw rows: 385234
usable rows: 363483
saved: ../data/processed/nyc_ems_2024q1_usable.parquet


In [20]:
df_check = pd.read_parquet("../data/processed/nyc_ems_2024q1_usable.parquet")
print(df_check.shape)
df_check.head()

(363483, 15)


,cad_incident_id,incident_datetime,initial_call_type,initial_severity_level_code,final_call_type,final_severity_level_code,valid_dispatch_rspns_time_indc,dispatch_response_seconds_qy,valid_incident_rspns_time_indc,incident_response_seconds_qy,incident_travel_tm_seconds_qy,held_indicator,incident_disposition_code,borough,zipcode
0,240010001,2024-01-01 00:00:03,INJURY,5,INJURY,5,Y,53,Y,393.0,340.0,N,93,QUEENS,11365
1,240010002,2024-01-01 00:00:14,CARD,3,CARD,3,Y,84,Y,430.0,346.0,N,82,BRONX,10457
2,240010004,2024-01-01 00:00:45,RESPIR,4,RESPIR,4,Y,1485,Y,1934.0,449.0,Y,82,MANHATTAN,10033
3,240010005,2024-01-01 00:00:55,BURNMI,4,BURNMI,4,Y,39,Y,1315.0,1276.0,N,82,BROOKLYN,11208
4,240010006,2024-01-01 00:01:12,EDPC,7,EDPC,7,Y,11,Y,1050.0,1039.0,N,82,RICHMOND / STATEN ISLAND,10309


## ACS Data Ingest 

### **ACS data has already been pulled and saved. Regenerate only if needed **


In [21]:
load_dotenv()
API_KEY = os.getenv("CENSUS_API_KEY")

if API_KEY is None:
    raise ValueError("CENSUS_API_KEY not found.")
    
print("API key loaded successfully.")

API key loaded successfully.


In [22]:
variables = {
    "B19013_001E": "median_income",
    "B02001_001E": "total_pop",
    "B02001_002E": "white_pop",
    "B02001_003E": "black_pop",
    "B03003_003E": "hispanic_pop",
}

base_url = "https://api.census.gov/data/2022/acs/acs5"

params = {
    "get": ",".join(variables.keys()),
    "for": "zip code tabulation area:*",
    "key": API_KEY
}

response = requests.get(base_url, params=params)
data = response.json()

acs = pd.DataFrame(data[1:], columns=data[0])

acs = acs.rename(columns=variables)
acs = acs.rename(columns={"zip code tabulation area": "zipcode"})

for col in variables.values():
    acs[col] = pd.to_numeric(acs[col], errors="coerce")

acs.head()

,median_income,total_pop,white_pop,black_pop,hispanic_pop,zipcode
0,17526,16834,14170,348,16726,00601
1,20260,37642,18479,555,35608,00602
2,17703,49075,36216,1719,48141,00603
3,19603,5590,3721,9,5552,00606
4,22796,25542,11182,576,24609,00610


In [23]:
acs["pct_black"] = acs["black_pop"] / acs["total_pop"]
acs["pct_white"] = acs["white_pop"] / acs["total_pop"]
acs["pct_hispanic"] = acs["hispanic_pop"] / acs["total_pop"]

acs.describe()

,median_income,total_pop,white_pop,black_pop,hispanic_pop,pct_black,pct_white,pct_hispanic
count,3.377400e+04,33774.000000,33774.000000,33774.000000,33774.000000,33187.000000,33187.000000,33187.000000
mean,-6.217092e+07,9900.097027,6500.458430,1230.967490,1924.154290,0.076271,0.790612,0.102644
std,1.939778e+08,14908.088606,9265.878654,3829.393787,5973.543684,0.156959,0.225438,0.172453
min,-6.666667e+08,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.846650e+04,649.250000,521.000000,0.000000,6.000000,0.000000,0.693711,0.007566
50%,6.388650e+04,2656.000000,2120.000000,29.000000,99.000000,0.009890,0.878071,0.037528
75%,8.277750e+04,13329.000000,8846.000000,610.750000,937.000000,0.066700,0.954793,0.109926
max,2.500010e+05,134008.000000,118381.000000,81608.000000,97669.000000,1.000000,1.000000,1.000000


In [24]:
#save full ACS pull (national ZCTA-level) to data/raw
os.makedirs("../data/raw", exist_ok=True)
acs_out = "../data/raw/acs_2022_acs5_zcta_full.parquet"
acs.to_parquet(acs_out, index=False)
print("saved:", acs_out)

saved: ../data/raw/acs_2022_acs5_zcta_full.parquet
